In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import chi2_contingency, f_oneway
import warnings
warnings.filterwarnings('ignore')

# 1. Тестирование гипотез

Для тестирования гипотез, сформулированных нами в ходе реализации разведочного анализа данных, загрузим датасет, получившийся по итогам предыдущего этапа:

In [2]:
df = pd.read_excel('/content/processed_df.xlsx')

In [3]:
df

,Family_Wealth,Home_Educaional_Resources,Home_Cultural_Resources,Mothers_Education,Fathers_Education,Highest_Parental_Education,Parents_emotional_support,Parents_learning_support,Teacher_support,Teacher_engagement,...,School_attitude,Persistence,Competitiveness,Neuroticism,Resilence,Motivation,Agreeableness,Сognitive_abilities,Weights,Grades
0,-0.273077,-0.343569,-0.271062,"ISCED 3A, ISCED 4",ISCED 5B,ISCED 5B,Strongly disagree,Strongly disagree,Strongly agree,Strongly disagree,...,-1.019794,0.490764,-0.708217,-1.979491,0.251532,-2.232942,0.997193,-0.288237,33.68177,4627.390333
1,1.744554,0.916468,1.027900,"ISCED 5A, 6",ISCED 5B,"ISCED 5A, 6",Agree,Agree,Agree,Agree,...,-0.268408,0.239738,-0.157990,-0.050583,-0.171202,0.712579,0.053661,0.067309,33.68177,4167.809667
2,0.939412,-1.532721,2.838671,"ISCED 5A, 6","ISCED 5A, 6","ISCED 5A, 6",Agree,Agree,Disagree,Disagree,...,-0.767082,0.239738,-0.221939,-0.050583,-1.031889,-1.179932,-1.596094,0.172182,33.68177,3517.277667
3,-1.319868,-0.526490,-0.130827,ISCED 5B,ISCED 5B,ISCED 5B,Agree,Agree,Disagree,Disagree,...,-0.268408,-0.919860,-0.708217,-0.604661,-1.070633,-2.232942,-0.262506,-0.901150,33.68177,3335.927000
4,-0.132970,-1.532721,1.027900,ISCED 5B,ISCED 5B,ISCED 5B,Agree,Strongly agree,Agree,Disagree,...,-0.260652,0.239738,2.143837,-0.050583,-0.171202,-1.800702,0.053661,-0.529703,33.68177,4003.476667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14599,0.443260,0.916468,2.838671,"ISCED 5A, 6","ISCED 5A, 6","ISCED 5A, 6",Agree,Strongly agree,Agree,Agree,...,0.403005,-0.524625,-0.221939,0.029840,0.251532,-0.133951,0.295387,0.613449,403.94359,4694.626333
14600,-1.504082,-0.087239,-1.006818,ISCED 5B,ISCED 5B,ISCED 5B,Strongly agree,Strongly agree,Disagree,Disagree,...,-0.268408,0.239738,-0.373061,0.029840,-0.171202,-0.133951,0.562141,-0.583381,403.94359,4345.704333
14601,-0.833150,-0.999539,-0.349288,ISCED 5B,ISCED 5B,ISCED 5B,Agree,Agree,Strongly disagree,Disagree,...,-0.767082,1.072615,2.143837,0.706148,-0.653510,-0.464470,-0.008450,0.765854,403.94359,5054.748000
14602,-0.004539,-1.532721,-0.653011,"ISCED 5A, 6",ISCED 5B,"ISCED 5A, 6",Strongly agree,Strongly agree,Agree,Agree,...,-0.268408,0.239738,0.258924,-0.604661,0.251532,0.712579,0.997193,0.765854,403.94359,4561.412667


Отберем из загруженного датасета те признаки, которые фигурируют в сформулированных нами гипотезах, а также переменную весов и таргет-переменную с результатами тестирования:

In [4]:
cols = ['Sex', 'Parents_emotional_support', 'Parents_learning_support', 'Teacher_support', 'Teacher_engagement', 'Grade_Repetition', 'Immigration_Status', 'Weights', 'Grades']
df_test = df[cols].copy()

Так как в получившемся датафрейме находятся данные, не разбитые на категории, произведем разбиение в соответствии с принципами разбиения на этапе разведочного анализа данных:

In [5]:
support_categories = ['Parents_emotional_support', 'Parents_learning_support',
                      'Teacher_support', 'Teacher_engagement']

for c in support_categories:
  df_test[c] = df_test[c].apply(lambda x: 'Высокая' if x in ['Agree', 'Strongly agree'] else 'Низкая')

In [6]:
df_test['Grade_Repetition'] = df_test['Grade_Repetition'].apply(lambda x: 'Повторял курс' if x == 'Repeated a  grade' else 'Не повторял курс')

In [7]:
df_test['Immigration_Status'] = df_test['Immigration_Status'].map({'Native': 'Коренной житель',
                                                                 'Second-Generation': 'Иммигрант второго поколения',
                                                                 'First-Generation': 'Иммигрант первого поколения'})

In [8]:
df_test['Immigrant_binary'] = df_test['Immigration_Status'].apply(lambda x: 'Коренной житель' if x == 'Коренной житель' else 'Иммигрант')

In [9]:
df_test['Sex'] = df_test['Sex'].apply(lambda x: 'Женский' if x == 'Female' else 'Мужской')

Мы подготовили данные для статистического тестирования, поэтому теперь мы можем переходить к следующему этапу.

Так как имеющиеся в основных статистических библиотеках функции не учитывают веса, возникает необходимость создания собственных функций, учитывающих данный показатель:

### 1. Функция `weighted_stats()`

**Назначение:** Данная функция будет использоваться для вычисления базовых взвешенных статистик для одной группы наблюдений.

**Методология расчета:**
- **Взвешенное среднее**: $\bar{x}_w = \frac{\sum_{i=1}^{n} w_i x_i}{\sum_{i=1}^{n} w_i}$, где $w_i$ — вес i-го наблюдения (вес респондента в выборке PISA)
- **Взвешенная дисперсия**: $s_w^2 = \frac{\sum_{i=1}^{n} w_i (x_i - \bar{x}_w)^2}{\sum_{i=1}^{n} w_i  - 1}$
- **Взвешенное стандартное отклонение**: $s_w = \sqrt{s_w^2}$

**Причина использования:** В данных PISA используется сложная стратифицированная кластерная выборка, где каждый респондент имеет свой вес. Игнорирование весов приводит к смещению оценок и нерепрезентативности результатов для генеральной совокупности российских школьников.

In [10]:
def weighted_stats(data, value_col='Grades', weight_col='Weights'):
    weights = data[weight_col].values
    values = data[value_col].values
    n = len(data)
    mean = np.average(values, weights=weights)
    sum_weights = np.sum(weights)
    variance = np.average((values - mean)**2, weights=weights) * (sum_weights / (sum_weights - 1))
    std = np.sqrt(variance)
    return mean, std, variance, n

### 2. Функция `weighted_ttest()`

**Назначение:** Проведение взвешенного t-теста для сравнения двух независимых групп, например, сравнения средних баллов тестирования для обучающихся женского и мужского пола.

**Методология расчета:**
Используется **критерий Уэлча (Welch's t-test)**, который не предполагает равенства дисперсий в сравниваемых группах:

$$t = \frac{\bar{x}_1 - \bar{x}_2}{\sqrt{\frac{s_1^2}{n_1} + \frac{s_2^2}{n_2}}}$$

Число степеней свободы рассчитывается по формуле Уэлча-Саттертвейта:

$$df \approx \frac{\left( \frac{s_1^2}{n_1} + \frac{s_2^2}{n_2} \right)^2}{\frac{(s_1^2/n_1)^2}{n_1-1} + \frac{(s_2^2/n_2)^2}{n_2-1}}$$

**Причина использования:** Выбранный тест не требует предположения о равенстве дисперсий, является более надежным в сравнении с классическим t-тестом Стьюдента, а также корректно работает при неравных размерах групп

In [11]:
def weighted_ttest(df, group_col, group1_val, group2_val, value_col='Grades', weight_col='Weights'):
    group1 = df[df[group_col] == group1_val]
    group2 = df[df[group_col] == group2_val]
    mean1, std1, var1, n1 = weighted_stats(group1, value_col, weight_col)
    mean2, std2, var2, n2 = weighted_stats(group2, value_col, weight_col)

    t_stat = (mean1 - mean2) / np.sqrt(var1/n1 + var2/n2)
    df_welch = (var1/n1 + var2/n2)**2 / ((var1/n1)**2/(n1-1) + (var2/n2)**2/(n2-1))
    p_value = 2 * stats.t.sf(np.abs(t_stat), df_welch)

    result = {
        'group1': group1_val, 'group2': group2_val,
        'mean1': mean1, 'mean2': mean2,
        'diff': mean1 - mean2,
        't_stat': t_stat,
        'p_value': p_value,
        'n1': n1, 'n2': n2
    }
    return result

### 3. Функция `weighted_anova()`

**Назначение:** Проведение взвешенного однофакторного дисперсионного анализа (ANOVA) для сравнения трех и более групп: в рассматриваемом случае ANOVA-тест будет проводиться для трех групп признака `Immigration_Status`, после чего будут проведены post-hoc тесты для выявления конкретных различающихся пар.

**Методология расчета:**
ANOVA реализован через взвешенную линейную регрессию с категориальным предиктором:

$$Grades = \beta_0 + \beta_1 \cdot I_{group2} + \beta_2 \cdot I_{group3} + \cdots + \varepsilon$$

где $I_{groupk}$ — индикаторная переменная (дамми-переменная) для k-й группы.

**F-статистика** рассчитывается как:

$$F = \frac{MS_{between}}{MS_{within}} = \frac{SS_{between} / (k-1)}{SS_{within} / (N-k)}$$

где:
- $SS_{between}$ — межгрупповая сумма квадратов
- $SS_{within}$ — внутригрупповая сумма квадратов
- $k$ — число групп
- $N$ — общее число наблюдений

**Причина использования:** Взвешенный ANOVA-тест позволяет одновременно сравнить несколько групп, является устойчивым к умеренным нарушениям нормальности при больших выборках и дает общий ответ о наличии различий между группами в целом

In [12]:
def weighted_anova(df, group_col, value_col='Grades', weight_col='Weights'):
    formula = f"{value_col} ~ C({group_col})"
    model = ols(formula, data=df).fit()

    f_stat = model.fvalue
    p_value = model.f_pvalue

    groups = df[group_col].unique()
    group_stats = []
    for group in groups:
        group_data = df[df[group_col] == group]
        mean, std, _, n = weighted_stats(group_data, value_col, weight_col)
        group_stats.append({'group': group, 'mean': mean, 'std': std, 'n': n})

    return {
        'f_stat': f_stat,
        'p_value': p_value,
        'r_squared': model.rsquared,
        'groups': groups,
        'group_stats': group_stats,
        'model': model
    }

### 4. Функция `weighted_cohens_d()`

**Назначение:** Расчет стандартизированной меры эффекта (effect size) для оценки практической значимости различий (оценки того, насколько велики обнаруженные различия).

**Методология расчета:**
Коэффициент Cohen's d рассчитывается как стандартизированная разница между средними двух групп:

$$d = \frac{|\bar{x}_1 - \bar{x}_2|}{s_{pooled}}$$

где $s_{pooled}$ — объединенное стандартное отклонение:

$$s_{pooled} = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1 + n_2 - 2}}$$

**Интерпретация значений Cohen's d:**
| Значение d | Величина эффекта |
|------------|------------------|
| d < 0.2    | Очень маленький  |
| 0.2 ≤ d < 0.5 | Маленький    |
| 0.5 ≤ d < 0.8 | Средний      |
| d ≥ 0.8    | Большой          |

**Причина использования:** p-value показывает статистическую значимость, но не говорит о практической значимости, вследствие чего возникает необходимость в
сравнении силы эффекта между разными переменными (например, влияние поддержки родителей vs влияние пола)

In [13]:
def weighted_cohens_d(df, group_col, group1_val, group2_val, value_col='Grades', weight_col='Weights'):
    group1 = df[df[group_col] == group1_val]
    group2 = df[df[group_col] == group2_val]

    mean1, _, var1, n1 = weighted_stats(group1, value_col, weight_col)
    mean2, _, var2, n2 = weighted_stats(group2, value_col, weight_col)

    pooled_var = ((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2)
    pooled_std = np.sqrt(pooled_var)

    cohens_d = abs(mean1 - mean2) / pooled_std

    if cohens_d < 0.2:
        effect = "Очень маленький"
    elif cohens_d < 0.5:
        effect = "Маленький"
    elif cohens_d < 0.8:
        effect = "Средний"
    else:
        effect = "Большой"

    return cohens_d, effect

### 5. Функция `weighted_posthoc_tests()`

**Назначение:** Проведение попарных сравнений между группами после ANOVA с применением поправки на множественные сравнения для детекции пар групп, которые статистически значимо различаются.

**Методология расчета:**
1. **Попарные сравнения:** Для каждой пары групп проводится взвешенный t-тест (Уэлча)
2. **Поправка Бонферрони:**
   $$\alpha_{corrected} = \frac{\alpha}{m}$$
   где $\alpha = 0.05$ (исходный уровень значимости), $m$ — число попарных сравнений
   
   Скорректированное p-value:
   $$p_{corrected} = \min(p_{original} \cdot m, 1)$$

**Причины использования:** при множественных сравнениях возрастает риск ошибки I рода, а поправка Бонферрони контролирует этот риск

In [14]:
def weighted_posthoc_tests(df, group_col, value_col='Grades', weight_col='Weights'):

    groups = df[group_col].unique()
    results = []
    comparisons = []

    for i in range(len(groups)):
        for j in range(i+1, len(groups)):
            group1, group2 = groups[i], groups[j]
            test_result = weighted_ttest(df, group_col, group1, group2, value_col, weight_col)
            comparisons.append(test_result)

    n_comparisons = len(comparisons)
    for res in comparisons:
        p_corrected = min(res['p_value'] * n_comparisons, 1.0)
        res['p_value_corrected'] = p_corrected
        res['significant'] = p_corrected < 0.05

    return comparisons

Мы создали все необходимые функции для расчета взвешенных статистик, вследствие чего можем переходить к осуществлению статистического тестирования выдвинутых гипотез на полученных данных. Для информативности и осуществления последующей визуализации будем сохранять результаты проведенных тестов в переменную **'all_results'**:

In [15]:
all_results = []

### **Гипотеза № 1: Обучающиеся женского пола в среднем демонстрируют более высокие результаты тестирования, чем обучающиеся мужского пола**

Основанием для выдвижения данной гипотезы стало не только достаточное количество научных работ, в которых фиксируется наличие гендерного разрыва в академической успеваемости, но и результаты проведенного разведывательного анализа. На визуализациях взвешенного распределения оценок по полу наблюдался более высокий пик плотности распределения у обучающихся женского пола, что может свидетельствовать о большей однородности их результатов и немного более высоких результатах по сравнению с обучающимися мужского пола.

In [16]:
result = weighted_ttest(df_test, 'Sex', 'Мужской', 'Женский')
cohens_d, effect = weighted_cohens_d(df_test, 'Sex', 'Мужской', 'Женский')

print(f"Среднее значение результатов тестирования обучающихся мужского пола равно {result['mean1']:.1f}")
print(f"Среднее значение результатов тестирования обучающихся женского пола равно {result['mean2']:.1f}")
print(f"Разница между средними баллами групп составляет {abs(result['diff']):.1f} баллов")
print(f"t-статистика проведенного теста равна {result['t_stat']:.4f}, p-value - {result['p_value']:.6f}")
print(f"Cohen's d равен {cohens_d:.3f} ({effect} эффект)")

Среднее значение результатов тестирования обучающихся мужского пола равно 4783.0
Среднее значение результатов тестирования обучающихся женского пола равно 4856.9
Разница между средними баллами групп составляет 73.9 баллов
t-статистика проведенного теста равна -5.6854, p-value - 0.000000
Cohen's d равен 0.094 (Очень маленький эффект)


Результаты проведенного взвешенного t-теста определили существование статистически значимых различий в академической успеваемости между школьниками и школьницами: средний балл девушек оказался на 73.9 пункта выше, чем у юношей. Несмотря на это, значение коэффициента Коэна оказалось равным 0.094, что в соответствии с установленной Коэном классификацией соответствует очень маленькому эффекту. Таким образом, несмотря на наличие статистически значимых различий, практическая значимость гендерного фактора для объяснения вариативности академической успеваемости является минимальной.

**Вывод: Гипотеза подтверждена статистически, но не подтверждена практически**

Добавим результаты проведенного тестирования в общий список результатов:

In [17]:
all_results.append({
    'Признак': 'Пол',
    'Сравнение': 'Женский / Мужской',
    'Разница': result['diff'],
    't_stat': result['t_stat'],
    'p_value': result['p_value'],
    'Сohens_d': cohens_d,
    'Размер эффекта': effect,
    'Количество групп': 2
})

### **Гипотеза № 2: Обучающиеся, получающие эмоциональную поддержку от родителей, в среднем демонстрируют более высокие результаты тестирования, чем обучающиеся лишенные поддержки**

Основанием для выдвижения данной гипотезы послужили визуализации, созданные в ходе реализации EDA: на графиках пик плотности распределения у группы учащихся, получающих поддержку, расположен выше, а медианное значение превышает соответствующий показатель группы, не получающей поддержку. Это позволяет предположить наличие позитивной связи между эмоциональной поддержкой родителей и результатами тестирования.

In [18]:
result = weighted_ttest(df_test, 'Parents_emotional_support', 'Высокая', 'Низкая')
cohens_d, effect = weighted_cohens_d(df_test, 'Parents_emotional_support', 'Высокая', 'Низкая')

print(f"Среднее значение результатов тестирования обучающихся, которым родители оказывают эмоциональную поддержку, равно {result['mean1']:.1f}")
print(f"Среднее значение результатов тестирования обучающихся, которым родители не оказывают эмоциональную поддержку, равно {result['mean2']:.1f}")
print(f"Разница между средними баллами групп составляет {abs(result['diff']):.1f} баллов")
print(f"t-статистика проведенного теста равна {result['t_stat']:.4f}, p-value - {result['p_value']:.6f}")
print(f"Cohen's d равен {cohens_d:.3f} ({effect} эффект)")

Среднее значение результатов тестирования обучающихся, которым родители оказывают эмоциональную поддержку, равно 4846.2
Среднее значение результатов тестирования обучающихся, которым родители не оказывают эмоциональную поддержку, равно 4688.5
Разница между средними баллами групп составляет 157.7 баллов
t-статистика проведенного теста равна 8.8794, p-value - 0.000000
Cohen's d равен 0.201 (Маленький эффект)



Результаты взвешенного t-теста зафиксировали статистические значимые различия в образовательной результативности между обучающимися, получающими и не получающими эмоциональную поддержку от родителей. При этом средний балл учащихся лишенных родительской эмоциональной поддержки оказался на 157.7 баллов ниже. В то же время коэффициент Коэна составил 0.021, что соответствует маленькому эффекту. Таким образом, гипотеза о положительном влиянии эмоциональной поддержки родителей на академическую успеваемость находит как статистическое, так и практическое подтверждение, однако размер эффекта указывает на то, что данный фактор является одним из многих, влияющих на образовательные результаты.

**Вывод: Гипотеза подтверждена статистически и практически**

Добавим результаты проведенного тестирования в общий список результатов:

In [19]:
all_results.append({
    'Признак': 'Родительская эмоциональная поддержка',
    'Сравнение': 'Высокая / Низкая',
    'Разница': result['diff'],
    't_stat': result['t_stat'],
    'p_value': result['p_value'],
    'Сohens_d': cohens_d,
    'Размер эффекта': effect,
    'Количество групп': 2
})

### **Гипотеза № 3: Обучающиеся, получающие образовательную поддержку от родителей, в среднем демонстрируют более высокие результаты тестирования, чем обучающиеся лишенные поддержки**

Причиной формулировки гипотезы послужили исследования, демонстрирующие ключевую роль родительского участия в образовательном процессе. Помощь с домашними заданиями и контроль успеваемости способствуют лучшему усвоению информации и повышают академическую мотивацию учащихся. Кроме того, рассмотрение графика взвешенного распределения оценок по уровню родительской учебной поддержки показало более выраженную разницу в медианах и пиках плотности распределения не только между группами, но и между видами родительской поддержки - образовательной и эмоциональной.

In [20]:
result = weighted_ttest(df_test, 'Parents_learning_support', 'Высокая', 'Низкая')
cohens_d, effect = weighted_cohens_d(df_test, 'Parents_learning_support', 'Высокая', 'Низкая')

print(f"Среднее значение результатов тестирования обучающихся, которым родители оказывают учебную поддержку, равно {result['mean1']:.1f}")
print(f"Среднее значение результатов тестирования обучающихся, которым родители не оказывают учебную поддержку, равно {result['mean2']:.1f}")
print(f"Разница между средними баллами групп составляет {abs(result['diff']):.1f} баллов")
print(f"t-статистика проведенного теста равна {result['t_stat']:.4f}, p-value - {result['p_value']:.6f}")
print(f"Cohen's d равен {cohens_d:.3f} ({effect} эффект)")

Среднее значение результатов тестирования обучающихся, которым родители оказывают учебную поддержку, равно 4861.4
Среднее значение результатов тестирования обучающихся, которым родители не оказывают учебную поддержку, равно 4581.2
Разница между средними баллами групп составляет 280.2 баллов
t-статистика проведенного теста равна 15.5324, p-value - 0.000000
Cohen's d равен 0.359 (Маленький эффект)


Результаты взвешенного t-теста выявили статистически значимые различия в академической успеваемости между обучающимися, получающими и не получающими учебную поддержку от родителей: средний балл учащихся, которым оказывается образовательная поддержка от родителей оказался на 280.2 балла выше. Коэффициент Коэна составил 0.359, что соответствует маленькому, но близкому к среднему эффекту. Таким образом, гипотеза о положительном влиянии учебной поддержки родителей на академические успехи находит как статистическое, так и практическое подтверждение. При этом важно отметить, что влияние учебной поддержки оказалось более выраженным, чем влияние эмоциональной поддержки, что подтверждает теоретическое предположение о ключевой роли непосредственного участия родителей в образовательном процессе.

**Вывод: Гипотеза подтверждена статистически и практически**

Добавим результаты проведенного тестирования в общий список результатов:

In [21]:
all_results.append({
    'Признак': 'Родительская учебная поддержка',
    'Сравнение': 'Высокая / Низкая',
    'Разница': result['diff'],
    't_stat': result['t_stat'],
    'p_value': result['p_value'],
    'Сohens_d': cohens_d,
    'Размер эффекта': effect,
    'Количество групп': 2
})

### **Гипотеза № 4: Обучающиеся, которым учителя оказывают эмоциональную поддержку, в среднем демонстрируют более высокие результаты тестирования, чем обучающиеся лишенные поддержки**

Беря во внимание тот факт, что эмоциональная поддержка учителями, проявляющаяся в оказании внимания к потребностям и эмоциям школьников, создании доверительной атмосферы на уроках, снижает академическую тревожность и увеличивает вовлеченность в образовательный процесс, можно предположить что она также оказывает положительное влияние на учебную результативность. Более того, распределение оценок по уровню учительской поддержки показало, что группа школьников, получающих поддержку от учителей, характеризуется более высоким пиком плотности распределения. Это может свидетельствовать о положительной связи между данным фактором и академической успеваемостью.

In [22]:
result = weighted_ttest(df_test, 'Teacher_support', 'Высокая', 'Низкая')
cohens_d, effect = weighted_cohens_d(df_test, 'Teacher_support', 'Высокая', 'Низкая')

print(f"Среднее значение результатов тестирования обучающихся, которым учителя оказывают эмоциональную поддержку, равно  {result['mean1']:.1f}")
print(f"Среднее значение результатов тестирования обучающихся, которым учителя не оказывают эмоциональную поддержку, равно {result['mean2']:.1f}")
print(f"Разница между средними баллами групп составляет {abs(result['diff']):.1f} баллов")
print(f"t-статистика проведенного теста равна {result['t_stat']:.4f}, p-value - {result['p_value']:.6f}")
print(f"Cohen's d равен {cohens_d:.3f} ({effect} эффект)")

Среднее значение результатов тестирования обучающихся, которым учителя оказывают эмоциональную поддержку, равно  4856.5
Среднее значение результатов тестирования обучающихся, которым учителя не оказывают эмоциональную поддержку, равно 4733.7
Разница между средними баллами групп составляет 122.8 баллов
t-статистика проведенного теста равна 8.6397, p-value - 0.000000
Cohen's d равен 0.157 (Очень маленький эффект)


Результаты взвешенного t-теста выявили статистически значимые различия в академической успеваемости между школьниками, получающими и не получающими эмоциональную поддержку от учителей. Разница в средних баллах между группами при этом составила 122.8 балла. Коэффициент Коэна оказался равен 0.157, что соответствует маленькому эффекту. Из этого следует, что, несмотря на статистическую значимость различий, практическая значимость эмоциональной поддержки учителей является минимальной.

**Вывод: Гипотеза подтверждена статистически, однако ее практическая значимость невелика**

Добавим результаты проведенного тестирования в общий список результатов:

In [23]:
all_results.append({
    'Признак': 'Учительская эмоциональная поддержка',
    'Сравнение': 'Высокая / Низкая',
    'Разница': result['diff'],
    't_stat': result['t_stat'],
    'p_value': result['p_value'],
    'Сohens_d': cohens_d,
    'Размер эффекта': effect,
    'Количество групп': 2
})

### **Гипотеза № 5: Обучающиеся, преподаватели которых характеризуются высокой вовлеченностью в учебный процесс, демонстрируют более высокие результаты тестирования**

Вопреки распространенному предположению о том, что вовлеченность учителя в образовательный процесс является значимым предиктором академических достижений, результаты проведенного разведывательного анализа данных не выявили выраженных различий в распределении баллов между группами учащихся с вовлеченными и невовлеченными преподавателями. Визуализация взвешенного распределения оценок по типу вовлеченности продемонстрировала визуально близкое расположение медианных значений для сравниваемых групп. Данное наблюдение способствует формулированию предположения о том, что фактор вовлеченности учителя не является значимым для детерминации академической успеваемости.

In [24]:
result = weighted_ttest(df_test, 'Teacher_engagement', 'Высокая', 'Низкая')
cohens_d, effect = weighted_cohens_d(df_test, 'Teacher_engagement', 'Высокая', 'Низкая')

print(f"Среднее значение результатов тестирования обучающихся преподавателей, вовлеченных в учебный процесс, равно {result['mean1']:.1f}")
print(f"Среднее значение результатов тестирования обучающихся преподавателей, не вовлеченных в учебный процесс, равно {result['mean2']:.1f}")
print(f"Разница между средними баллами групп составляет {abs(result['diff']):.1f} баллов")
print(f"t-статистика проведенного теста равна {result['t_stat']:.4f}, p-value - {result['p_value']:.6f}")
print(f"Cohen's d равен {cohens_d:.3f} ({effect} эффект)")

Среднее значение результатов тестирования обучающихся преподавателей, вовлеченных в учебный процесс, равно 4836.8
Среднее значение результатов тестирования обучающихся преподавателей, не вовлеченных в учебный процесс, равно 4782.1
Разница между средними баллами групп составляет 54.7 баллов
t-статистика проведенного теста равна 3.8870, p-value - 0.000102
Cohen's d равен 0.070 (Очень маленький эффект)


Результаты взвешенного t-теста выявили статистически значимые различия в академической успеваемости между группами учащихся, обучающихся у вовлеченных и невовлеченных преподавателей. Средний балл учащихся, чьи учителя характеризуются высокой вовлеченностью оказался на 54.7 балла выше. Однако коэффициент Коэна составил всего 0.070, что в соответствии с классификацией соответствует очень маленькому эффекту. Столь низкое значение размера эффекта, позволяет заключить, что практическая значимость вовлеченности учителя для академической успеваемости является незначительной.

**Вывод: Гипотеза подтверждена статистически, практическая значимость крайне незначительна**

Добавим результаты проведенного тестирования в общий список результатов:

In [25]:
all_results.append({
    'Признак': 'Учительская вовлеченность',
    'Сравнение': 'Высокая / Низкая',
    'Разница': result['diff'],
    't_stat': result['t_stat'],
    'p_value': result['p_value'],
    'Сohens_d': cohens_d,
    'Размер эффекта': effect,
    'Количество групп': 2
})

### **Гипотеза № 6: Обучающиеся, остававшиеся на второй год, склоны к демонстрации более низких учебных результатов**

Согласно логике, учащиеся, оставленные на второй год, уже испытывали академические трудности в предшествующих периодах, следовательно, они могут иметь проблемы с освоением школьной программы и в настоящий момент. Кроме того, факт оставления на второй год оказывает негативное психологическое воздействие, снижая самооценку и учебную мотивацию учащихся, в результате чего потенциально понижается и вероятность академического успеха. Результаты разведочного анализа данных подкрепляют это предположение: график демонстрирует значительное смещение распределения в сторону более низких баллов у группы учащихся, повторявших курс, относительно группы, не повторявшей курс. Соответствующее наблюдение позволяет выдвинуть предположение о наличии статистически значимых различий между группами.

In [26]:
result = weighted_ttest(df_test, 'Grade_Repetition', 'Повторял курс', 'Не повторял курс')
cohens_d, effect = weighted_cohens_d(df_test, 'Grade_Repetition', 'Повторял курс', 'Не повторял курс')

print(f"Среднее значение результатов тестирования обучающихся, остававшихся на второй год, равно {result['mean1']:.1f}")
print(f"Среднее значение результатов тестирования обучающихся, не остававшихся на второй год, равно {result['mean2']:.1f}")
print(f"Разница между средними баллами групп составляет {abs(result['diff']):.1f} баллов")
print(f"t-статистика проведенного теста равна {result['t_stat']:.4f}, p-value - {result['p_value']:.6f}")
print(f"Cohen's d равен {cohens_d:.3f} ({effect} эффект)")

Среднее значение результатов тестирования обучающихся, остававшихся на второй год, равно 4055.5
Среднее значение результатов тестирования обучающихся, не остававшихся на второй год, равно 4833.2
Разница между средними баллами групп составляет 777.7 баллов
t-статистика проведенного теста равна -13.9051, p-value - 0.000000
Cohen's d равен 0.998 (Большой эффект)


Результаты взвешенного t-теста выявили статистически значимые различия в академической успеваемости между обучающимися, повторявшими и не повторявшими курс. Средний балл учащихся, не остававшихся на второй год, оказался на 777.7 балла выше, чем у учащихся, повторявших курс. Коэффициент Коэна составил 0.998, что соответствует большому наблюдаемому эффекту. Данное значение по совместительству является наибольшим среди всех проанализированных факторов, что свидетельствует о критической роли практики повторения курса в детерминации образовательных результатов.

**Вывод: Гипотеза подтверждена статистически и практически**

Добавим результаты проведенного тестирования в общий список результатов:

In [27]:
all_results.append({
    'Признак': 'Повторение курса',
    'Сравнение': 'Повторял / Не повторял',
    'Разница': result['diff'],
    't_stat': result['t_stat'],
    'p_value': result['p_value'],
    'Сohens_d': cohens_d,
    'Размер эффекта': effect,
    'Количество групп': 2
})

### **Гипотеза № 7.1: Иммиграционный статус обучающегося связан с его академической успеваемостью**

Статус миграции является важным социально-демографическим фактором, который может оказывать влияние на образовательные результаты учащихся. Обучающиеся из семей мигрантов могут сталкиваться с дополнительными трудностями в процессе обучения: языковым барьером, необходимостью адаптации к новой образовательной системе и культурной среде. В то же время некоторые исследования показывают, что дети мигрантов могут демонстрировать высокую мотивацию к обучению, рассматривая образование как механизм социальной мобильности. Таким образом, наличие неоднозначности в определении направления влияния отражается в формулировке гипотезы

In [28]:
anova_result = weighted_anova(df_test, 'Immigration_Status')

for stat in anova_result['group_stats']:
    print(f"Среднее значение результатов тестирования обучающихся в группе '{stat['group']}' равно {stat['mean']:.1f}")
print(f"F-статистика проведенного теста равна {anova_result['f_stat']:.4f}")
print(f"p-value проведенного теста равен {anova_result['p_value']:.6f}")
print(f"R-squared проведенного теста равен {anova_result['r_squared']:.4f}")

Среднее значение результатов тестирования обучающихся в группе 'Коренной житель' равно 4824.1
Среднее значение результатов тестирования обучающихся в группе 'Иммигрант первого поколения' равно 4595.6
Среднее значение результатов тестирования обучающихся в группе 'Иммигрант второго поколения' равно 4874.7
F-статистика проведенного теста равна 10.9970
p-value проведенного теста равен 0.000017
R-squared проведенного теста равен 0.0015


Результаты взвешенного однофакторного дисперсионного анализа (ANOVA) выявили статистически значимые различия в академической успеваемости между группами обучающихся с разным иммиграционным статусом. Средние баллы по группам "Коренной житель", "Иммигрант I-го поколения", "Иммигрант II-го поколения" составили 4824.1, 4595.6, 4874.7 баллов соотвественно. Наибольшая разница средних была отмечена между группой обучающихся, являющихся иммигрантами в первом поколении, и местными обучающимися. Значение p-value оказалось равно 0.000017, поэтому нулевая гипотеза об отсутствии различий между группами была отвергнута. Однако значение R², равное 0.0015, свидетельствует о том, что миграционный статус объясняет лишь 0.15% дисперсии результатов тестирования.  Таким образом, несмотря на статистическую значимость, практическая значимость миграционного статуса как фактора, определяющего академическую успеваемость, является крайне незначительной.

**Вывод: Гипотеза подтверждена статистически, но не практически**

Для выявления конкретных различающихся пар групп были проведем post-hoc попарные сравнения с поправкой Бонферрони:

In [29]:
posthoc_results = weighted_posthoc_tests(df_test, 'Immigration_Status')

for res in posthoc_results:
    print(f"{res['group1']} против {res['group2']}: "
          f"Разница между средними баллами групп составляет {abs(res['diff']):.1f} баллов, "
          f"p-value проведенного теста равен {res['p_value']:.6f}")

Коренной житель против Иммигрант первого поколения: Разница между средними баллами групп составляет 228.5 баллов, p-value проведенного теста равен 0.000000
Коренной житель против Иммигрант второго поколения: Разница между средними баллами групп составляет 50.6 баллов, p-value проведенного теста равен 0.183115
Иммигрант первого поколения против Иммигрант второго поколения: Разница между средними баллами групп составляет 279.1 баллов, p-value проведенного теста равен 0.000000


Полученные результаты свидетельствуют о том, что статистически значимые различия средних баллов тестирования выявлены для пары "Корреной житель" и "Иммигрант I-го поколения", а также для пары "Иммигрант II-го поколения" и "Иммигрант I-го поколения". Наблюдаемые различия можно обосновать трудностями адаптации к новой образовательной системе, различиями в культурной среде и социальных нормах, а также пробелами в знаниях, обусловленных различием образовательных программ. С этими сложностями сталкивается большинство мигрантов в первом поколении, самостоятельно сменивших страну проживания.

### **Гипотеза № 7.2: Обучающиеся-коренные жители демонстрируют более высокие академические результаты, чем обучающиеся-иммигранты**

Результаты предыдущего анализа (ANOVA и post-hoc тестов) показали, что иммигранты первого поколения демонстрируют статистически значимо более низкие результаты по сравнению как с коренными жителями, так и с иммигрантами второго поколения. В то же время результаты иммигрантов второго поколения статистически знаичмо не отличаются от коренных жителей. Данная разнонаправленная динамика не позволяет однозначно предсказать, сохранится ли разрыв при объединении двух иммиграционных подгрупп. Вследствие этого необходимо проверить, наблюдаются ли статистически и практически значимые разницы в результатах тестирования для местных школьников и школьников-иммигрантов (в любом из поколений)

In [30]:
result_binary = weighted_ttest(df_test, 'Immigrant_binary', 'Коренной житель', 'Иммигрант')
cohens_d_binary, effect_binary = weighted_cohens_d(df_test, 'Immigrant_binary', 'Коренной житель', 'Иммигрант')

print(f"Среднее значение результатов тестирования обучающихся-коренных жителей равно {result_binary['mean1']:.1f}")
print(f"Среднее значение результатов тестирования обучающихся-иммигрантов равно {result_binary['mean2']:.1f}")
print(f"Разница между средними баллами групп составляет {abs(result_binary['diff']):.1f} баллов")
print(f"t-статистика проведенного теста равна {result_binary['t_stat']:.4f}, p-value - {result_binary['p_value']:.6f}")
print(f"Cohen's d равен {cohens_d_binary:.3f} ({effect_binary} эффект)")

Среднее значение результатов тестирования обучающихся-коренных жителей равно 4824.1
Среднее значение результатов тестирования обучающихся-иммигрантов равно 4761.5
Разница между средними баллами групп составляет 62.6 баллов
t-статистика проведенного теста равна 2.1921, p-value - 0.028619
Cohen's d равен 0.080 (Очень маленький эффект)


Результаты бинарного взвешенного t-теста выявили статистически значимые различия в академической успеваемости между коренными жителями и иммигрантами. Средний балл коренных жителей оказался на 62.6 балла выше, чем средний балл иммигрантов. Однако коэффициент Коэна составил всего 0.080, что соответствует очень маленькому эффекту. Данный размера эффекта означает, что миграционный статус объясняет менее 1% дисперсии результатов тестирования. Таким образом, несмотря на статистическую значимость различий, практическая значимость миграционного статуса как фактора-предиктора является крайне незначительной.

**Вывод: Гипотеза подтверждена статистически, практическая значимость крайне незначительна**

Добавим результаты проведенных тестирований в общий список результатов:

In [31]:
all_results.append({
    'Признак': 'Иммиграционный статус (3 группы)',
    'Сравнение': 'Коренной / Иммигрант I-го поколения / Иммигрант II-го поколения',
    'Разница': None,
    't_stat': anova_result['f_stat'],
    'p_value': anova_result['p_value'],
    'Сohens_d': None,
    'Размер эффекта': None,
    'Количество групп': 3
})

In [32]:
all_results.append({
    'Признак': 'Иммиграционный статус (2 группы)',
    'Сравнение': 'Коренной / Иммигрант',
    'Разница': result_binary['diff'],
    't_stat': result_binary['t_stat'],
    'p_value': result_binary['p_value'],
    'Сohens_d': cohens_d_binary,
    'Размер эффекта': effect_binary,
    'Количество групп': 2
    })

При проведении статистического анализа данных с тестированием большого количества гипотез возникает проблема множественных сравнений. Суть данной проблемы заключается в том, что при многократной проверке статистических гипотез на одних и тех же данных возрастает вероятность совершения ошибки I рода. Для нивелирования риска совершения ошибки указанного типа предполагается использовать поправку Бонферрони, позволяющую повысить валидность статистических выводов в рамках настоящего исследования. Отберем результаты тестирований бинарных переменных и произведем перерасчет p-value с учетом поправки Бонферрони:

In [33]:
binary_results = [r for r in all_results if r['Количество групп'] == 2]
n_tests = len(binary_results)

for res in binary_results:
    p_corrected = min(res['p_value'] * n_tests, 1.0)
    res['p_value_corrected'] = p_corrected
    if p_corrected < 0.05:
        conclusion = "Отвергаем H0"
        interpretation = "Разница статистически значима"
    else:
        conclusion = "Не отвергаем H0"
        interpretation = "Нет статистически значимых различий"
    print(f"{res['Признак']} ({res['Сравнение']}):")
    print(f"Исходный p-value равен {res['p_value']:.6f}")
    print(f"Скорректированный p-value равен {p_corrected:.6f}")
    print(f"Вывод: {conclusion} → {interpretation}")
    print(f"Cohen's d равен {res['Сohens_d']:.3f} ({res['Размер эффекта']} эффект)")
    print()

Пол (Женский / Мужской):
Исходный p-value равен 0.000000
Скорректированный p-value равен 0.000000
Вывод: Отвергаем H0 → Разница статистически значима
Cohen's d равен 0.094 (Очень маленький эффект)

Родительская эмоциональная поддержка (Высокая / Низкая):
Исходный p-value равен 0.000000
Скорректированный p-value равен 0.000000
Вывод: Отвергаем H0 → Разница статистически значима
Cohen's d равен 0.201 (Маленький эффект)

Родительская учебная поддержка (Высокая / Низкая):
Исходный p-value равен 0.000000
Скорректированный p-value равен 0.000000
Вывод: Отвергаем H0 → Разница статистически значима
Cohen's d равен 0.359 (Маленький эффект)

Учительская эмоциональная поддержка (Высокая / Низкая):
Исходный p-value равен 0.000000
Скорректированный p-value равен 0.000000
Вывод: Отвергаем H0 → Разница статистически значима
Cohen's d равен 0.157 (Очень маленький эффект)

Учительская вовлеченность (Высокая / Низкая):
Исходный p-value равен 0.000102
Скорректированный p-value равен 0.000716
Вывод: Отвер

Применение поправки Бонферрони привело к изменению статуса проверки для гипотезы о влиянии иммиграционного статуса. Исходный p-value, равный 0.028619, позволял отвергнуть нулевую гипотезу, однако после коррекции p-value составил 0.200335, что превышает порог значимости в 0.05. Таким образом, с учетом множественных сравнений статистически значимые различия между коренными жителями и иммигрантами отсутствуют. Для гипотезы о вовлеченности учителей коррекция привела к увеличению p-value с 0.000102 до 0.000716, однако оба значения остались ниже 0.05, поэтому статус гипотезы не изменился, различия остаются статистически значимыми. Остальные гипотезы сохранили свой статус после коррекции.


Выведем для показа зафиксированные результаты тестирований и произведем резюмирование итогов по этапу тестирования гипотез:

In [35]:
pd.DataFrame(all_results)

,Признак,Сравнение,Разница,t_stat,p_value,Сohens_d,Размер эффекта,Количество групп,p_value_corrected
0,Пол,Женский / Мужской,-73.878386,-5.685404,1.330135e-08,0.094126,Очень маленький,2,9.310946e-08
1,Родительская эмоциональная поддержка,Высокая / Низкая,157.691940,8.879424,1.049032e-18,0.201177,Маленький,2,7.343222e-18
2,Родительская учебная поддержка,Высокая / Низкая,280.182261,15.532418,2.033284e-52,0.359474,Маленький,2,1.423298e-51
3,Учительская эмоциональная поддержка,Высокая / Низкая,122.792762,8.639710,6.700385e-18,0.156675,Очень маленький,2,4.690269e-17
4,Учительская вовлеченность,Высокая / Низкая,54.716870,3.886966,1.022787e-04,0.069678,Очень маленький,2,7.159511e-04
5,Повторение курса,Повторял / Не повторял,-777.679330,-13.905118,1.689799e-31,0.997872,Большой,2,1.182859e-30
6,Иммиграционный статус (3 группы),Коренной / Иммигрант I-го поколения / Иммигран...,NaN,10.996979,1.689142e-05,NaN,None,3,NaN
7,Иммиграционный статус (2 группы),Коренной / Иммигрант,62.626348,2.192075,2.861929e-02,0.079729,Очень маленький,2,2.003350e-01


# Заключение

Проведенный статистический анализ позволяет сделать следующие выводы:

- Наиболее сильное влияние на академическую успеваемость в исследуемой выборке российских школьников оказывает фактор повторения курса, который демонстрирует большой размер эффекта (d = 0.998);

- Семейные факторы — учебная и эмоциональная поддержка родителей — также показывают статистически и практически значимые, хотя и небольшие, положительные эффекты;


- Факторы, связанные с преподавателями (эмоциональная поддержка и вовлеченность) демонстрируют статистическую значимость при очень маленьком или отсутствующем практическом эффекте;

- Несмотря на достижение статистической значимости, переменные пола обучающегося и его иммиграционного статуса имеют низкие значения коэффициента Коэна, вследствие чего не вносят существенного вклада в объяснение вариативности академической успеваемости.